In [13]:
import os
import pyreadstat
import pandas as pd
import numpy as np

# Paths to files
school_file = r'C:\Users\ASUS\pisa_project\data\CY09_MS_SCH_PUF\CY09_MS_SCH_PUF.sav'
student_file = r'C:\Users\ASUS\pisa_project\data\STU_QQQ_SAS\CY08MSP_STU_QQQ.SAS7BDAT'

Бібліотеки успішно підключено!


In [22]:
# Key factors fof the school from codebook
school_features = [
    'CNT', 'CNTSCHID',         # keys for association (country and school ID)
    'CLSIZE', 'STRATIO',        # class size and student-teacher ratio
    'PROATCE',                  # proportion of fully certified teachers
    'EDUSHORT', 'STAFFSHORT',   # equipment and personnel storage index
    'SCHLTYPE'                  # type of school(public or private)
]

df_school, _ = pyreadstat.read_sav(school_file, usecols=school_features)
print(df_school.shape[0], 'schools', df_school.shape[1], 'columns')
df_school.head(3)

25133 schools 7 columns


,CNT,CNTSCHID,CLSIZE,PROATCE,SCHLTYPE,EDUSHORT,STAFFSHORT
0,ALB,00800001,18.0,5.0,3.0,-0.0703,0.0053
1,ALB,00800002,NaN,NaN,NaN,NaN,NaN
2,ALB,00800003,18.0,5.0,3.0,-0.6481,-0.7828


In [23]:
# Selecting student perfomance indicators and control variables
student_features = [
    'CNT', 'CNTSCHID',        # Ключі зв'язку
    'ESCS',                   # family socio-economic status index
    'W_FSTUWT',               # final statistical weight of a student
    'PV1MATH', 'PV2MATH',     # (Plausible Values)
    'PV3MATH', 'PV4MATH', 'PV5MATH'
]

df_student, _ = pyreadstat.read_sas7bdat(student_file, usecols=student_features)
print(df_student.shape[0], 'students',df_student.shape[1], 'columns')
df_student.head(3)

613744 students 9 columns


,CNT,CNTSCHID,ESCS,W_FSTUWT,PV1MATH,PV2MATH,PV3MATH,PV4MATH,PV5MATH
0,ALB,800282.0,1.1112,3.15874,179.583,201.334,222.248,194.943,222.755
1,ALB,800115.0,-3.0507,4.34524,308.247,304.444,316.129,234.353,323.448
2,ALB,800242.0,-0.1867,7.83860,267.514,343.743,327.157,290.771,316.227


In [25]:
# Selecting countries
target_countries = ['DEU', 'FRA', 'POL', 'EST', 'ITA', 'ESP', 'UKR']

df_school_sub = df_school[df_school['CNT'].isin(target_countries)].copy()
df_student_sub = df_student[df_student['CNT'].isin(target_countries)].copy()

# making keys same float/str to avoid join errors
df_student_sub['CNTSCHID'] = df_student_sub['CNTSCHID'].astype(float)
df_school_sub['CNTSCHID'] = df_school_sub['CNTSCHID'].astype(float)
df_student_sub['CNT'] = df_student_sub['CNT'].astype(str)
df_school_sub['CNT'] = df_school_sub['CNT'].astype(str)

# join school parameters with student
merged_df = pd.merge(
    df_student_sub,
    df_school_sub,
    on=['CNT', 'CNTSCHID'],
    how='inner'
)

print(merged_df.shape[0], 'rows', merged_df.shape[1], 'columns')

61707 rows 14 columns


In [26]:
#Data cleaning
# 1. remove critical gaps
merged_df = merged_df.dropna(subset=['W_FSTUWT', 'PV1MATH'])

# 2. calculating avarage math score
pv_cols = ['PV1MATH', 'PV2MATH', 'PV3MATH', 'PV4MATH', 'PV5MATH']
merged_df['MATH_SCORE'] = merged_df[pv_cols].mean(axis=1)

# 3. list of factors to fill in with meadian by country
impute_vars = ['ESCS', 'STRATIO', 'STRATIO_Q', 'PROATCE', 'EDUSHORT', 'STAFFSHORT', 'STAFFSHO']

# fill in only those columns that are in the table
existing_impute = [col for col in impute_vars if col in merged_df.columns]

for col in existing_impute:
    merged_df[col] = merged_df.groupby('CNT')[col].transform(lambda x: x.fillna(x.median()))

# 4. remove gaps only in those key fields that are present
required_cols = ['ESCS', 'PROATCE', 'EDUSHORT']
if 'STRATIO_Q' in merged_df.columns:
    required_cols.append('STRATIO_Q')
elif 'STRATIO' in merged_df.columns:
    required_cols.append('STRATIO')

merged_df = merged_df.dropna(subset=required_cols)

print(f"Number of records prepared for modeling: {len(merged_df)}")
print("Avaliable columns in dataset:", list(merged_df.columns))

Number of records prepared for modeling: 61707
Avaliable columns in dataset: ['CNT', 'CNTSCHID', 'ESCS', 'W_FSTUWT', 'PV1MATH', 'PV2MATH', 'PV3MATH', 'PV4MATH', 'PV5MATH', 'CLSIZE', 'PROATCE', 'SCHLTYPE', 'EDUSHORT', 'STAFFSHORT', 'MATH_SCORE']


In [27]:
output_dir = r'C:\Users\ASUS\pisa_project\data\processed'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'pisa_cleaned_dataset.parquet')
merged_df.to_parquet(output_path, index=False)

print(f"File is ready and save in the path:\n{output_path}")
merged_df.head()

File is ready and save in the path:
C:\Users\ASUS\pisa_project\data\processed\pisa_cleaned_dataset.parquet


,CNT,CNTSCHID,ESCS,W_FSTUWT,PV1MATH,PV2MATH,PV3MATH,PV4MATH,PV5MATH,CLSIZE,PROATCE,SCHLTYPE,EDUSHORT,STAFFSHORT,MATH_SCORE
0,EST,23300149.0,0.9668,4.69717,527.654,590.326,563.005,549.312,546.229,13.0,3.0,3.0,-1.4210,0.5795,555.3052
1,EST,23300171.0,-0.3947,1.72897,715.893,719.525,738.151,742.997,732.032,23.0,4.0,3.0,-0.3123,0.6736,729.7196
2,EST,23300075.0,-0.9071,1.76996,433.490,420.050,406.021,428.302,405.179,23.0,5.0,3.0,0.2681,1.8545,418.6084
3,EST,23300194.0,-0.6361,2.48957,472.419,475.092,468.743,458.075,457.784,13.0,4.0,2.0,1.3063,1.6255,466.4226
4,EST,23300079.0,-0.9308,1.43473,449.444,413.698,399.089,416.500,380.016,13.0,5.0,3.0,-0.0703,0.5120,411.7494
